In [2]:

import argparse
import hashlib
import re
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import GroupShuffleSplit

LABELS = {"GPT": 0, "Claude": 1, "Gemini": 2}
COLUMNS = ["prompt_id", "LLM_Input", "LLM_output", "LLM_name", "label"]


def family(name):
    """Only assign families that can be inferred unambiguously from model IDs."""
    name = str(name or "").lower()
    if name.startswith(("gpt-", "chatgpt-")):
        return "GPT"
    if name.startswith("claude-"):
        return "Claude"
    if name.startswith("gemini-"):
        return "Gemini"
    return None


def message_text(message):
    """Return plain text when every content block is text-only."""
    if not isinstance(message, dict):
        return None
    content = message.get("content")
    if isinstance(content, str):
        return content.strip() or None
    if not isinstance(content, list) or not content:
        return None
    parts = []
    for block in content:
        if not isinstance(block, dict) or block.get("type") != "text":
            return None
        # Even type=text can carry an image attachment in the Arena schema.
        if block.get("image") is not None or block.get("mimeType") is not None:
            return None
        value = block.get("text")
        if not isinstance(value, str):
            return None
        parts.append(value)
    return "\n".join(parts).strip() or None


def single_turn(conversation):
    """Strictly one user message followed by one assistant answer."""
    if not isinstance(conversation, list) or len(conversation) != 2:
        return None
    user, assistant = conversation
    if not isinstance(user, dict) or not isinstance(assistant, dict):
        return None
    if user.get("role") != "user" or assistant.get("role") != "assistant":
        return None
    prompt, response = message_text(user), message_text(assistant)
    return (prompt, response) if prompt and response else None


def prompt_key(prompt):
    canonical = re.sub(r"\s+", " ", prompt).strip().casefold()
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()


def extract_rows(rows, max_rows=20000, language="en"):
    records = []
    scanned = 0
    for record in rows:
        if max_rows and scanned >= max_rows:
            break
        scanned += 1
        if language and record.get("language") != language:
            continue
        conv_a = single_turn(record.get("conversation_a"))
        conv_b = single_turn(record.get("conversation_b"))
        if conv_a is None or conv_b is None:
            continue
        pa, ra = conv_a
        pb, rb = conv_b
        if prompt_key(pa) != prompt_key(pb):
            continue
        key = prompt_key(pa)
        for name, answer in ((record.get("model_a"), ra), (record.get("model_b"), rb)):
            group = family(name)
            if group:
                records.append({
                    "prompt_id": key,
                    "LLM_Input": pa,
                    "LLM_output": answer,
                    "LLM_name": group,
                    "label": LABELS[group],
                })
    print(f"Arena battles scanned: {scanned:,}; target responses extracted: {len(records):,}")
    return pd.DataFrame.from_records(records, columns=COLUMNS)


def make_splits(df, seed=42):
    """Group before balancing so repeated prompts cannot cross splits."""
    df = df.drop_duplicates(subset=["prompt_id", "LLM_name", "LLM_output"]).copy()
    if any((df.LLM_name == name).sum() < 3 for name in LABELS):
        raise ValueError("Insufficient examples for at least one family. Increase --max-rows.")
    split1 = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=seed)
    train_idx, rest_idx = next(split1.split(df, groups=df.prompt_id))
    train, rest = df.iloc[train_idx], df.iloc[rest_idx]
    split2 = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=seed)
    val_idx, test_idx = next(split2.split(rest, groups=rest.prompt_id))
    splits = {"train": train, "validation": rest.iloc[val_idx], "test": rest.iloc[test_idx]}
    # Balance separately within each split, rather than before prompt-group splitting.
    for split_name, part in splits.items():
        counts = part.LLM_name.value_counts().reindex(LABELS.keys(), fill_value=0)
        n = int(counts.min())
        if n == 0:
            raise ValueError(f"{split_name}: missing a class; try more input rows or a different seed")
        splits[split_name] = (
            part.groupby("LLM_name", group_keys=False)
            .sample(n=n, random_state=seed)
            .sample(frac=1, random_state=seed)
            .reset_index(drop=True)[COLUMNS]
        )
    keys = {name: set(part.prompt_id) for name, part in splits.items()}
    assert keys["train"].isdisjoint(keys["validation"])
    assert keys["train"].isdisjoint(keys["test"])
    assert keys["validation"].isdisjoint(keys["test"])
    return splits


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--max-rows", type=int, default=20000,
                        help="Arena battles to scan; 0 means entire dataset")
    parser.add_argument("--output-dir", default="arena_processed")
    parser.add_argument("--language", default="en", help="Use 'all' for any language")
    args = parser.parse_args(args=[])
    out = Path(args.output_dir)
    out.mkdir(parents=True, exist_ok=True)
    dataset = load_dataset("lmarena-ai/arena-human-preference-140k", split="train", streaming=True)
    df = extract_rows(dataset, max_rows=args.max_rows,
                      language=None if args.language == "all" else args.language)
    if df.empty:
        raise ValueError("No usable examples; check dataset availability and filters.")
    print("Extracted counts before balancing:\n", df.LLM_name.value_counts().to_string())
    splits = make_splits(df)
    for name, part in splits.items():
        path = out / f"{name}.csv"
        part.to_csv(path, index=False, encoding="utf-8")
        print(f"{path}: {len(part):,} rows; each family={len(part)//3:,}")
    combined = pd.concat(splits.values(), ignore_index=True)
    combined.to_csv(out / "arena_processed_all.csv", index=False, encoding="utf-8")
    print("Final columns:", list(combined.columns))
    print("Saved:", out.resolve())


if __name__ == "__main__":
    main()


README.md:   0%|          | 0.00/5.54k [00:00<?, ?B/s]

Arena battles scanned: 20,000; target responses extracted: 8,006
Extracted counts before balancing:
 LLM_name
Claude    3756
Gemini    2802
GPT       1448
arena_processed/train.csv: 3,021 rows; each family=1,007
arena_processed/validation.csv: 624 rows; each family=208
arena_processed/test.csv: 693 rows; each family=231
Final columns: ['prompt_id', 'LLM_Input', 'LLM_output', 'LLM_name', 'label']
Saved: /content/arena_processed
